# Den n8n-Workflow als Python-Code verstehen
# *Understanding the n8n workflow as Python code*

Dieses Notebook übersetzt die beiden n8n-Workflows des Projekts in normales Python:<br>
*This notebook translates the two n8n workflows of the project into plain Python:*

1. **Advisory Input** – Eingang normalisieren und mit `processing_status = NEW` speichern.<br>
   *__Advisory Input__ – normalise the input and store it with `processing_status = NEW`.*
2. **Advisory Processing** – ältesten offenen Eingang übernehmen, strukturieren, validieren, mit dem Asset-Inventar abgleichen, Status bestimmen und Ergebnis speichern.<br>
   *__Advisory Processing__ – claim the oldest open request, structure it, validate it, match it against the asset inventory, decide the status and store the result.*

Die Umsetzung folgt diesen Dateien im Repository:<br>
*It follows these files in the repository:*

- [`docs/architecture.de.md`](../docs/architecture.de.md)<br>
  *[`docs/architecture.md`](../docs/architecture.md)*
- [`workflows/01-advisory-input.json`](../workflows/01-advisory-input.json)
- [`workflows/02-advisory-processing.json`](../workflows/02-advisory-processing.json)
- [`data/synthetic_security_advisories.md`](../data/synthetic_security_advisories.md)

Wichtig: Das Notebook ist eine **didaktische, lokal ausführbare Übersetzung**. PostgreSQL wird durch einen In-Memory-Speicher ersetzt. Der optionale Groq-Aufruf ist enthalten, wird aber ohne API-Key nicht ausgeführt. Die fachliche Entscheidungslogik entspricht dem n8n-Workflow.<br>
*Note: the notebook is a __didactic translation that runs locally__. PostgreSQL is replaced by an in-memory store. The optional Groq call is included but is not executed without an API key. The decision logic is the same as in the n8n workflow.*

## 1. Zuordnung: n8n-Node → Python
## *1. Mapping: n8n node → Python*

| n8n | Python in diesem Notebook<br>*Python in this notebook* |
|---|---|
| Webhook + Edit Fields + PostgreSQL Insert | `InMemoryRepository.accept()` |
| SQL: ältesten `NEW`-Datensatz übernehmen<br>*SQL: claim the oldest `NEW` request* | `claim_next()` |
| Switch `structured` / `unstructured` | `extract_advisory_facts()` |
| Information Extractor + Groq | `groq_extract()` oder reproduzierbares LLM-Fixture<br>*`groq_extract()` or a reproducible LLM fixture* |
| Validate Advisory Schema | `validate_advisory()` |
| Load Asset Inventory | `ASSET_INVENTORY` |
| Match Products and Versions | `match_products_and_versions()` |
| Decide Status | `decide_status()` |
| Save Processing Result | `save_result()` |
| Prepare/Save LLM Error | `save_error()` |

Die Trennung bleibt dieselbe wie in n8n:<br>
*The separation is the same as in n8n:*

> Das LLM extrahiert Fakten aus Freitext. Python-Regeln entscheiden über Asset-Treffer und Status.<br>
> *The LLM extracts facts from free text. Python rules decide on asset matches and status.*

## 2. Setup
## *2. Setup*

Das Notebook benötigt für den lokalen Prototyp nur die Python-Standardbibliothek. Dadurch läuft es in VS Code ohne zusätzliche Pakete.<br>
*The notebook only needs the Python standard library, so it runs in VS Code or Jupyter without additional packages.*

In [1]:
from __future__ import annotations

from copy import deepcopy
from datetime import datetime, timezone
import json
import os
import re
from pprint import pprint
from typing import Any, Callable, Optional


def utc_now() -> str:
    """UTC-Zeitstempel im ISO-Format."""
    return datetime.now(timezone.utc).isoformat()


ALLOWED_PAYLOAD_TYPES = {"structured", "unstructured"}
ALLOWED_VERSION_RULES = {"exact", "range", "before", "through", "all", "unknown"}

B_FIELDS = [
    "is_security_advisory",
    "advisory_reference",
    "cve_ids",
    "affected_products",
    "severity_stated_by_source",
    "recommended_action",
    "missing_information",
    "evidence_quotes",
]

print("Setup geladen.")

Setup geladen.


## 3. Synthetisches Asset-Inventar
## *3. Synthetic asset inventory*

Das sind die sieben Test-Assets aus dem Repository. Eine Zeile steht für eine konkrete Softwareinstallation. Im echten Workflow werden diese Daten aus PostgreSQL geladen.<br>
*These are the seven test assets of the repository. Each row is one software installation. In the real workflow this data is loaded from PostgreSQL.*

In [2]:
ASSET_INVENTORY = [
    {
        "asset_id": "A-REM-01",
        "customer_name": "Kunde A – Werkzeugbau",
        "device_name": "Fernwartungsserver 1",
        "vendor": "Musterwerk Systems GmbH",
        "product": "MusterLink Remote Gateway",
        "installed_version": "2.3.1",
    },
    {
        "asset_id": "A-REM-02",
        "customer_name": "Kunde A – Werkzeugbau",
        "device_name": "Fernwartungsserver 2",
        "vendor": "Musterwerk Systems GmbH",
        "product": "MusterLink Remote Gateway",
        "installed_version": "2.3.2",
    },
    {
        "asset_id": "A-BKP-01",
        "customer_name": "Kunde A – Werkzeugbau",
        "device_name": "Sicherungsserver",
        "vendor": "SicherWerk Software AG",
        "product": "DataSafe Backup",
        "installed_version": "6.1",
    },
    {
        "asset_id": "B-REM-01",
        "customer_name": "Kunde B – Verpackungstechnik",
        "device_name": "Fernwartungsserver",
        "vendor": "Musterwerk Systems GmbH",
        "product": "MusterLink Remote Gateway",
        "installed_version": "2.4.0",
    },
    {
        "asset_id": "B-HMI-01",
        "customer_name": "Kunde B – Verpackungstechnik",
        "device_name": "Produktionsrechner",
        "vendor": "Anlagenblick GmbH",
        "product": "PlantView HMI",
        "installed_version": "7.4",
    },
    {
        "asset_id": "C-REM-01",
        "customer_name": "Kunde C – Medizintechnik-Zulieferer",
        "device_name": "Fernwartungsserver",
        "vendor": "Musterwerk Systems GmbH",
        "product": "MusterLink Remote Gateway",
        "installed_version": "2.3.0",
    },
    {
        "asset_id": "C-ERP-01",
        "customer_name": "Kunde C – Medizintechnik-Zulieferer",
        "device_name": "Verwaltungsserver",
        "vendor": "Warenstrom Software GmbH",
        "product": "WarenFlow ERP",
        "installed_version": "3.8",
    },
]

print(f"{len(ASSET_INVENTORY)} Assets geladen.")

7 Assets geladen.


## 4. Workflow 1: Eingang annehmen und speichern
## *4. Workflow 1: accept and store the input*

In n8n erzeugen `Webhook` und `Edit Fields` die Eingangsfelder. PostgreSQL speichert den Datensatz mit `NEW`. Die HTTP-Antwort lautet anschließend `202 Accepted`.<br>
*In n8n, `Webhook` and `Edit Fields` create the input fields. PostgreSQL stores the record with `NEW`. The HTTP response is then `202 Accepted`.*

Hier übernimmt `InMemoryRepository` dieselbe Rolle. Der Speicher ist absichtlich einfach: Er zeigt die Zustände, ersetzt aber weder PostgreSQL-Transaktionen noch `FOR UPDATE SKIP LOCKED`.<br>
*Here `InMemoryRepository` plays the same role. It is deliberately simple: it shows the states but does not replace PostgreSQL transactions or `FOR UPDATE SKIP LOCKED`.*

In [3]:
class InMemoryRepository:
    """Lokaler Ersatz für public.advisory_requests."""

    def __init__(self) -> None:
        self.rows: dict[str, dict[str, Any]] = {}
        self._sequence = 0

    def accept(
        self,
        *,
        source_channel: str,
        payload_type: str,
        raw_text: str,
    ) -> dict[str, Any]:
        if payload_type not in ALLOWED_PAYLOAD_TYPES:
            raise ValueError(f"Unzulässiger payload_type: {payload_type!r}")
        if not isinstance(raw_text, str) or not raw_text.strip():
            raise ValueError("raw_text muss ein nicht leerer String sein.")

        self._sequence += 1
        request_id = f"REQ-PY-{self._sequence:04d}"
        row = {
            "request_id": request_id,
            "received_at": utc_now(),
            "source_channel": source_channel,
            "payload_type": payload_type,
            "raw_text": raw_text,
            "processing_status": "NEW",
            "attempt_count": 0,
            "schema_version": 1,
            "error_detail": None,
            "processed_at": None,
        }
        self.rows[request_id] = row

        return {
            "http_status": 202,
            "accepted": True,
            "request_id": request_id,
            "processing_status": "NEW",
        }

    def claim_next(self) -> Optional[dict[str, Any]]:
        """Übernimmt den ältesten NEW-Datensatz.

        In PostgreSQL erledigt dies der Workflow atomar mit
        FOR UPDATE SKIP LOCKED. Diese lokale Variante ist nicht parallelitätssicher.
        """
        candidates = [
            row for row in self.rows.values()
            if row["processing_status"] == "NEW"
        ]
        if not candidates:
            return None

        row = min(
            candidates,
            key=lambda value: (value["received_at"], value["request_id"]),
        )
        row["processing_status"] = "PROCESSING"
        row["attempt_count"] += 1
        return deepcopy(row)

    def save_result(self, result: dict[str, Any]) -> dict[str, Any]:
        request_id = result["request_id"]
        row = self.rows[request_id]
        fields_to_store = B_FIELDS + [
            "extraction_method",
            "model_provider",
            "model_name",
            "status",
            "matched_assets",
            "reason_codes",
            "decided_at",
            "schema_valid",
            "evidence_valid",
            "validation_errors",
            "normalization_notes",
        ]
        for field in fields_to_store:
            if field in result:
                row[field] = deepcopy(result[field])

        row["processing_status"] = "DONE"
        row["error_detail"] = None
        row["processed_at"] = utc_now()
        return deepcopy(row)

    def save_error(self, request_id: str, error_detail: str) -> dict[str, Any]:
        row = self.rows[request_id]
        row["processing_status"] = "ERROR"
        row["error_detail"] = error_detail
        row["processed_at"] = utc_now()
        return deepcopy(row)


repository = InMemoryRepository()
print("Leerer lokaler Eingangsspeicher angelegt.")

Leerer lokaler Eingangsspeicher angelegt.


## 5. Extraktionspfad: strukturiert oder unstrukturiert
## *5. Extraction path: structured or unstructured*

- `structured`: `raw_text` enthält bereits JSON; Python ordnet die acht Felder aus Bereich B direkt zu.<br>
  *`structured`: `raw_text` already contains JSON; Python maps the eight fields of group B directly.*
- `unstructured`: Ein LLM darf ausschließlich Bereich B erzeugen. Es erhält weder das Asset-Inventar noch die spätere Statuslogik.<br>
  *`unstructured`: an LLM may only produce group B. It receives neither the asset inventory nor the status logic.*

Für einen reproduzierbaren Lauf verwenden wir unten ein gespeichertes LLM-Ergebnis. Das ist kein Ersatz für das Modell, sondern ein stabiles Test-Fixture. Der optionale echte Groq-Aufruf ist anschließend als Funktion enthalten und wird nicht automatisch gestartet.<br>
*For a reproducible run, a stored LLM result is used below. It does not replace the model; it is a stable test fixture. The optional real Groq call follows as a function and is not started automatically.*

In [4]:
def extract_advisory_facts(
    request: dict[str, Any],
    *,
    llm_output: Optional[dict[str, Any]] = None,
) -> dict[str, Any]:
    if request["payload_type"] == "structured":
        parsed = json.loads(request["raw_text"])
        facts = {field: parsed[field] for field in B_FIELDS}
        return {
            **facts,
            "extraction_method": "structured_mapping",
            "model_provider": None,
            "model_name": None,
        }

    if request["payload_type"] == "unstructured":
        if llm_output is None:
            raise RuntimeError(
                "LLM extraction failed: Für den Offline-Lauf fehlt ein LLM-Fixture."
            )
        facts = {field: deepcopy(llm_output[field]) for field in B_FIELDS}
        return {
            **facts,
            "extraction_method": "llm",
            "model_provider": "groq",
            "model_name": "openai/gpt-oss-20b",
        }

    raise ValueError(f"Unbekannter payload_type: {request['payload_type']!r}")

### Optional: echter Groq-Aufruf
### *Optional: real Groq call*

Diese Funktion ist absichtlich nicht Teil des automatischen Notebook-Laufs. Sie benötigt:<br>
*This function is deliberately not part of the automatic run. It needs:*

- `pip install openai`
- Umgebungsvariable `GROQ_API_KEY`<br>
  *environment variable `GROQ_API_KEY`*
- Netzwerkzugriff<br>
  *network access*

Der Rückgabewert wird anschließend trotzdem durch denselben Python-Validator geprüft. Ein Modellfehler darf dadurch nicht direkt zu einem fachlichen Status werden.<br>
*Its output still goes through the same Python validator, so a model error cannot turn directly into a status.*

In [5]:
EXTRACTION_SYSTEM_PROMPT = """
Du extrahierst ausschließlich Advisory-Fakten aus dem Originaltext.
Gib genau die Felder aus Bereich B als JSON aus und erzeuge keine Status-,
Asset- oder Audit-Felder. Erfinde keine Informationen. Fehlende Einzelwerte
sind null. evidence_quotes müssen nicht leer sein und wortwörtlich im
Originaltext vorkommen. missing_information enthält nur Angaben, die für
Produkt- oder Versionsbetroffenheit entscheidungsrelevant fehlen, unklar
oder widersprüchlich sind.
""".strip()


def groq_extract(raw_text: str) -> dict[str, Any]:
    """Optionaler Live-Aufruf; wird in diesem Notebook nicht automatisch ausgeführt."""
    try:
        from openai import OpenAI
    except ImportError as exc:
        raise RuntimeError("Zuerst `pip install openai` ausführen.") from exc

    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Umgebungsvariable GROQ_API_KEY fehlt.")

    client = OpenAI(
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1",
    )
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        temperature=0,
        max_tokens=3000,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
            {"role": "user", "content": raw_text},
        ],
    )
    return json.loads(response.choices[0].message.content)

## 6. Schema- und Belegprüfung
## *6. Schema and evidence check*

Diese Funktion ist eng an den Python-Code im n8n-Node `Validate Advisory Schema` angelehnt. Sie prüft:<br>
*This function closely follows the Python code in the n8n node `Validate Advisory Schema`. It checks:*

- Pflichtfelder und Datentypen,<br>
  *required fields and data types,*
- zulässige Versionsregeln,<br>
  *allowed version rules,*
- passende Grenzen und Inklusivitätswerte,<br>
  *consistent bounds and inclusivity flags,*
- nicht leere Belegstellen, die wirklich in `raw_text` vorkommen,<br>
  *non-empty evidence quotes that really occur in `raw_text`,*
- zwei kontrollierte Normalisierungen.<br>
  *two controlled normalisations.*

Ein technischer Schema- oder Belegfehler führt später zu `REVIEW_REQUIRED`, nicht zu einer erfundenen Entwarnung.<br>
*A schema or evidence error later leads to `REVIEW_REQUIRED`, never to an invented all-clear.*

In [6]:
REQUIRED_PRODUCT_FIELDS = [
    "vendor",
    "product",
    "affected_version_raw",
    "version_rule_type",
    "lower_bound",
    "lower_inclusive",
    "upper_bound",
    "upper_inclusive",
    "fixed_version",
]


def is_string_or_none(value: Any) -> bool:
    return isinstance(value, str) or value is None


def is_boolean_or_none(value: Any) -> bool:
    return isinstance(value, bool) or value is None


def has_text(value: Any) -> bool:
    return isinstance(value, str) and value.strip() != ""


def validate_advisory(original_data: dict[str, Any]) -> dict[str, Any]:
    data = deepcopy(original_data)
    schema_errors: list[str] = []
    normalization_notes: list[str] = []

    for field in B_FIELDS:
        if field not in data:
            schema_errors.append(f"MISSING_FIELD:{field}")

    if (
        "is_security_advisory" in data
        and not isinstance(data["is_security_advisory"], bool)
    ):
        schema_errors.append("INVALID_TYPE:is_security_advisory")

    for field in [
        "advisory_reference",
        "severity_stated_by_source",
        "recommended_action",
    ]:
        if field in data and not is_string_or_none(data[field]):
            schema_errors.append(f"INVALID_TYPE:{field}")

    for field in ["cve_ids", "missing_information", "evidence_quotes"]:
        if field in data:
            value = data[field]
            if (
                not isinstance(value, list)
                or not all(isinstance(entry, str) for entry in value)
            ):
                schema_errors.append(f"INVALID_TYPE:{field}")

    if "affected_products" in data and not isinstance(
        data["affected_products"], list
    ):
        schema_errors.append("INVALID_TYPE:affected_products")

    if (
        data.get("is_security_advisory") is True
        and isinstance(data.get("affected_products"), list)
        and len(data["affected_products"]) == 0
    ):
        schema_errors.append("MISSING_VALUE:affected_products")

    if isinstance(data.get("affected_products"), list):
        for index, product in enumerate(data["affected_products"]):
            if not isinstance(product, dict):
                schema_errors.append(
                    f"INVALID_TYPE:affected_products[{index}]"
                )
                continue

            for field in REQUIRED_PRODUCT_FIELDS:
                if field not in product:
                    schema_errors.append(
                        f"MISSING_FIELD:affected_products[{index}].{field}"
                    )

            for field in [
                "vendor",
                "product",
                "affected_version_raw",
                "lower_bound",
                "upper_bound",
                "fixed_version",
            ]:
                if field in product and not is_string_or_none(product[field]):
                    schema_errors.append(
                        f"INVALID_TYPE:affected_products[{index}].{field}"
                    )

            for field in ["lower_inclusive", "upper_inclusive"]:
                if field in product and not is_boolean_or_none(product[field]):
                    schema_errors.append(
                        f"INVALID_TYPE:affected_products[{index}].{field}"
                    )

            rule = product.get("version_rule_type")
            if (
                "version_rule_type" in product
                and (
                    not isinstance(rule, str)
                    or rule not in ALLOWED_VERSION_RULES
                )
            ):
                schema_errors.append(
                    f"INVALID_VALUE:affected_products[{index}].version_rule_type"
                )

            if not has_text(product.get("product")):
                schema_errors.append(
                    f"MISSING_VALUE:affected_products[{index}].product"
                )

            lower_present = has_text(product.get("lower_bound"))
            upper_present = has_text(product.get("upper_bound"))

            # Normalisierung 1: "PlantView HMI 7.4" -> "PlantView HMI"
            if rule == "exact" and lower_present and has_text(product.get("product")):
                product_name = product["product"].strip()
                exact_version = product["lower_bound"].strip()
                version_suffix = f" {exact_version}"
                if product_name.endswith(version_suffix):
                    normalized_product = product_name[:-len(version_suffix)].strip()
                    if normalized_product:
                        product["product"] = normalized_product
                        normalization_notes.append(
                            "NORMALIZED_PRODUCT_VERSION_SUFFIX:"
                            f"affected_products[{index}]:"
                            f"{product_name}->{normalized_product}"
                        )

            # Normalisierung 2: zwei Grenzen bedeuten einen Bereich.
            if (
                isinstance(rule, str)
                and rule in ALLOWED_VERSION_RULES
                and lower_present
                and upper_present
                and rule != "range"
            ):
                old_rule = rule
                product["version_rule_type"] = "range"
                rule = "range"
                normalization_notes.append(
                    "NORMALIZED_VERSION_RULE:"
                    f"affected_products[{index}]:{old_rule}->range"
                )

            if rule == "range":
                if not lower_present or not upper_present:
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}].range_requires_both_bounds"
                    )
                if (
                    not isinstance(product.get("lower_inclusive"), bool)
                    or not isinstance(product.get("upper_inclusive"), bool)
                ):
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}]."
                        "range_requires_inclusive_flags"
                    )
            elif rule == "exact":
                if not lower_present:
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}].exact_requires_lower_bound"
                    )
                if product.get("lower_inclusive") is not True:
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}]."
                        "exact_requires_lower_inclusive_true"
                    )
            elif rule == "before":
                if not upper_present:
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}].before_requires_upper_bound"
                    )
                if product.get("upper_inclusive") is not False:
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}]."
                        "before_requires_upper_inclusive_false"
                    )
            elif rule == "through":
                if not upper_present:
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}].through_requires_upper_bound"
                    )
                if product.get("upper_inclusive") is not True:
                    schema_errors.append(
                        "INCONSISTENT_VERSION_RULE:"
                        f"affected_products[{index}]."
                        "through_requires_upper_inclusive_true"
                    )

    raw_text = data.get("raw_text")
    evidence_quotes = data.get("evidence_quotes")
    evidence_valid = (
        isinstance(raw_text, str)
        and isinstance(evidence_quotes, list)
        and len(evidence_quotes) > 0
        and all(
            isinstance(quote, str)
            and quote.strip() != ""
            and quote in raw_text
            for quote in evidence_quotes
        )
    )

    validation_errors = list(schema_errors)
    if not evidence_valid:
        validation_errors.append("EVIDENCE_NOT_IN_RAW_TEXT")

    data["schema_valid"] = len(schema_errors) == 0
    data["evidence_valid"] = evidence_valid
    data["validation_errors"] = validation_errors
    data["normalization_notes"] = normalization_notes
    return data

## 7. Produkt- und Versionsabgleich
## *7. Product and version matching*

Produkt und Hersteller werden nur für den Vergleich getrimmt und kleingeschrieben. Es gibt kein unscharfes Namensmatching.<br>
*Product and vendor are trimmed and lower-cased for the comparison only. There is no fuzzy name matching.*

Punktversionen wie `2.3.1` werden segmentweise verglichen. Fehlende Segmente zählen als `0`, also ist `2.3` gleich `2.3.0`. Nicht interpretierbare Versionen liefern `None`; das bedeutet „unklar“ und wird nicht geraten.<br>
*Dotted versions such as `2.3.1` are compared segment by segment. Missing segments count as `0`, so `2.3` equals `2.3.0`. Versions that cannot be interpreted return `None`, meaning "unclear"; nothing is guessed.*

In [7]:
def normalize_name(value: Any) -> str:
    return str(value or "").strip().lower()


def version_parts(value: Any) -> Optional[list[int]]:
    text = str(value or "").strip()
    if not re.fullmatch(r"\d+(?:\.\d+)*", text):
        return None
    return [int(part) for part in text.split(".")]


def compare_versions(left: Any, right: Any) -> Optional[int]:
    left_parts = version_parts(left)
    right_parts = version_parts(right)
    if left_parts is None or right_parts is None:
        return None

    length = max(len(left_parts), len(right_parts))
    left_parts += [0] * (length - len(left_parts))
    right_parts += [0] * (length - len(right_parts))

    if left_parts < right_parts:
        return -1
    if left_parts > right_parts:
        return 1
    return 0


def version_matches(
    installed_version: str,
    affected: dict[str, Any],
) -> Optional[bool]:
    rule_type = affected.get("version_rule_type")

    if rule_type == "all":
        return True
    if rule_type == "unknown":
        return None

    if rule_type == "exact":
        target = affected.get("lower_bound") or affected.get("upper_bound")
        if not target:
            match = re.search(
                r"\d+(?:\.\d+)*",
                str(affected.get("affected_version_raw") or ""),
            )
            target = match.group(0) if match else None
        comparison = compare_versions(installed_version, target)
        return None if comparison is None else comparison == 0

    if rule_type == "before":
        comparison = compare_versions(installed_version, affected.get("upper_bound"))
        return None if comparison is None else comparison < 0

    if rule_type == "through":
        comparison = compare_versions(installed_version, affected.get("upper_bound"))
        return None if comparison is None else comparison <= 0

    if rule_type == "range":
        lower_comparison = compare_versions(
            installed_version,
            affected.get("lower_bound"),
        )
        upper_comparison = compare_versions(
            installed_version,
            affected.get("upper_bound"),
        )
        if lower_comparison is None or upper_comparison is None:
            return None

        lower_matches = (
            lower_comparison >= 0
            if affected.get("lower_inclusive")
            else lower_comparison > 0
        )
        upper_matches = (
            upper_comparison <= 0
            if affected.get("upper_inclusive")
            else upper_comparison < 0
        )
        return lower_matches and upper_matches

    return None


def match_products_and_versions(
    data: dict[str, Any],
    assets: list[dict[str, Any]],
) -> dict[str, Any]:
    product_match = False
    version_check_unclear = False
    matched_assets: dict[str, dict[str, Any]] = {}

    for affected in data.get("affected_products", []):
        for asset in assets:
            same_product = (
                normalize_name(affected.get("product"))
                == normalize_name(asset.get("product"))
            )
            same_vendor = (
                not affected.get("vendor")
                or normalize_name(affected.get("vendor"))
                == normalize_name(asset.get("vendor"))
            )
            if not same_product or not same_vendor:
                continue

            product_match = True
            matches = version_matches(asset["installed_version"], affected)

            if matches is None:
                version_check_unclear = True
            elif matches is True:
                matched_assets[asset["asset_id"]] = {
                    "asset_id": asset["asset_id"],
                    "inventory_product": asset["product"],
                    "installed_version": asset["installed_version"],
                    "match_reason": (
                        f"Installierte Version {asset['installed_version']} erfüllt "
                        f"die Regel „{affected.get('affected_version_raw') or affected.get('version_rule_type')}“."
                    ),
                }

    return {
        **data,
        "product_match": product_match,
        "version_check_unclear": version_check_unclear,
        "matched_assets": list(matched_assets.values()),
        "match_count": len(matched_assets),
    }

## 8. Statusentscheidung und Gesamt-Orchestrierung
## *8. Status decision and orchestration*

Die Reihenfolge ist wichtig:<br>
*The order matters:*

1. `is_security_advisory = false` → `INVALID_ADVISORY`
2. Schema oder Belege ungültig → `REVIEW_REQUIRED`<br>
   *schema or evidence invalid → `REVIEW_REQUIRED`*
3. Echtes, gültiges Advisory → erst Produkt- und Versionsabgleich<br>
   *real, valid advisory → product and version matching first*
4. Produkt nicht im Inventar → `NO_MATCH_FOUND`<br>
   *product not in the inventory → `NO_MATCH_FOUND`*
5. Produkt vorhanden, Version aber unklar → `REVIEW_REQUIRED`<br>
   *product present, but version unclear → `REVIEW_REQUIRED`*
6. mindestens ein Versions-Treffer → `POTENTIALLY_AFFECTED`<br>
   *at least one version match → `POTENTIALLY_AFFECTED`*
7. Produkt vorhanden, Version nicht betroffen → `NO_MATCH_FOUND`<br>
   *product present, version not affected → `NO_MATCH_FOUND`*

Damit bleibt `NM-04` korrekt `NO_MATCH_FOUND`: Die Version ist zwar unbekannt, aber schon das Produkt ist im Inventar nicht vorhanden.<br>
*This keeps `NM-04` at `NO_MATCH_FOUND`: the version is unknown, but the product is not in the inventory at all.*

In [8]:
def set_invalid_advisory_status(data: dict[str, Any]) -> dict[str, Any]:
    return {
        **data,
        "status": "INVALID_ADVISORY",
        "matched_assets": [],
        "reason_codes": ["NOT_A_SECURITY_ADVISORY"],
        "decided_at": utc_now(),
    }


def set_validation_review_status(data: dict[str, Any]) -> dict[str, Any]:
    reason_codes: list[str] = []
    if data.get("schema_valid") is not True:
        reason_codes.append("SCHEMA_VALIDATION_FAILED")
    if data.get("evidence_valid") is not True:
        reason_codes.append("EVIDENCE_VALIDATION_FAILED")

    return {
        **data,
        "status": "REVIEW_REQUIRED",
        "matched_assets": [],
        "reason_codes": reason_codes,
        "decided_at": utc_now(),
    }


def decide_status(data: dict[str, Any]) -> dict[str, Any]:
    has_relevant_missing_information = any(
        isinstance(entry, str) and entry.strip()
        for entry in data.get("missing_information", [])
    )

    if not data.get("product_match"):
        status = "NO_MATCH_FOUND"
        reason_codes = ["PRODUCT_NOT_IN_INVENTORY"]
    elif has_relevant_missing_information or data.get("version_check_unclear"):
        status = "REVIEW_REQUIRED"
        reason_codes = ["VERSION_INFORMATION_UNCLEAR"]
    elif data.get("match_count", 0) > 0:
        status = "POTENTIALLY_AFFECTED"
        reason_codes = ["PRODUCT_AND_VERSION_MATCH"]
    else:
        status = "NO_MATCH_FOUND"
        reason_codes = ["PRODUCT_FOUND_VERSION_NOT_AFFECTED"]

    return {
        **data,
        "status": status,
        "matched_assets": (
            [] if status == "REVIEW_REQUIRED"
            else data.get("matched_assets", [])
        ),
        "match_count": (
            0 if status == "REVIEW_REQUIRED"
            else data.get("match_count", 0)
        ),
        "reason_codes": reason_codes,
        "decided_at": utc_now(),
    }


def process_next_request(
    repository: InMemoryRepository,
    *,
    assets: list[dict[str, Any]],
    llm_outputs: Optional[dict[str, dict[str, Any]]] = None,
) -> dict[str, Any]:
    request = repository.claim_next()
    if request is None:
        return {
            "work_available": False,
            "result_code": "NO_NEW_REQUEST",
            "message": (
                "Kein Datensatz mit processing_status = NEW gefunden. "
                "Der Datensatz wird möglicherweise bereits verarbeitet "
                "oder ist schon abgeschlossen."
            ),
        }

    try:
        llm_output = (llm_outputs or {}).get(request["request_id"])
        facts = extract_advisory_facts(request, llm_output=llm_output)
        validated = validate_advisory({**request, **facts})

        # Routing-Priorität wie im n8n-Switch:
        if validated.get("is_security_advisory") is False:
            decision = set_invalid_advisory_status(validated)
        elif (
            validated.get("schema_valid") is not True
            or validated.get("evidence_valid") is not True
        ):
            decision = set_validation_review_status(validated)
        else:
            matched = match_products_and_versions(validated, assets)
            decision = decide_status(matched)

        return repository.save_result(decision)

    except Exception as exc:
        return repository.save_error(
            request["request_id"],
            str(exc) or "LLM extraction failed",
        )

## 9. Ende-zu-Ende-Demo mit `PA-01`
## *9. End-to-end demo with `PA-01`*

Der Herstellerhinweis nennt MusterLink Remote Gateway 2.3.0 bis einschließlich 2.3.2. Das reproduzierbare LLM-Fixture enthält nur extrahierte Fakten und wörtliche Belegstellen. Danach arbeiten ausschließlich feste Python-Regeln.<br>
*The advisory names MusterLink Remote Gateway 2.3.0 up to and including 2.3.2. The reproducible LLM fixture only contains extracted facts and verbatim evidence quotes. From there on, only fixed Python rules are at work.*

In [9]:
PA01_RAW_TEXT = """Von: security-notifications@musterwerk.example
Betreff: Sicherheitsinformation MS-2026-014 – MusterLink Remote Gateway
Meldungstyp: Security Advisory
Herstellerbewertung: Hoch

Bei einer internen Sicherheitsprüfung wurde ein Fehler in der Anmeldekomponente von MusterLink Remote Gateway festgestellt. Unter bestimmten Umständen könnte dadurch ein unbefugter Zugriff auf das Fernwartungssystem ermöglicht werden.

Betroffen sind die Versionen 2.3.0 bis einschließlich 2.3.2. Installationen ab Version 2.4.0 sind nach unserem derzeitigen Kenntnisstand nicht betroffen. Wir empfehlen, vorhandene Installationen zeitnah zu überprüfen und betroffene Systeme auf Version 2.4.0 zu aktualisieren.

Eine CVE-Nummer wurde bislang nicht vergeben. Diese Nachricht wurde automatisch über den Sicherheitsverteiler der Musterwerk Systems GmbH versendet."""

PA01_LLM_OUTPUT = {
    "is_security_advisory": True,
    "advisory_reference": "MS-2026-014",
    "cve_ids": [],
    "affected_products": [
        {
            "vendor": "Musterwerk Systems GmbH",
            "product": "MusterLink Remote Gateway",
            "affected_version_raw": "Versionen 2.3.0 bis einschließlich 2.3.2",
            "version_rule_type": "range",
            "lower_bound": "2.3.0",
            "lower_inclusive": True,
            "upper_bound": "2.3.2",
            "upper_inclusive": True,
            "fixed_version": "2.4.0",
        }
    ],
    "severity_stated_by_source": "Hoch",
    "recommended_action": (
        "Vorhandene Installationen zeitnah überprüfen und betroffene "
        "Systeme auf Version 2.4.0 aktualisieren."
    ),
    "missing_information": [],
    "evidence_quotes": [
        "Bei einer internen Sicherheitsprüfung wurde ein Fehler in der Anmeldekomponente von MusterLink Remote Gateway festgestellt.",
        "Betroffen sind die Versionen 2.3.0 bis einschließlich 2.3.2.",
        "Installationen ab Version 2.4.0 sind nach unserem derzeitigen Kenntnisstand nicht betroffen.",
    ],
}

demo_repository = InMemoryRepository()
accepted = demo_repository.accept(
    source_channel="email",
    payload_type="unstructured",
    raw_text=PA01_RAW_TEXT,
)
request_id = accepted["request_id"]

result_pa01 = process_next_request(
    demo_repository,
    assets=ASSET_INVENTORY,
    llm_outputs={request_id: PA01_LLM_OUTPUT},
)

summary_pa01 = {
    "request_id": result_pa01["request_id"],
    "processing_status": result_pa01["processing_status"],
    "status": result_pa01["status"],
    "reason_codes": result_pa01["reason_codes"],
    "matched_asset_ids": [
        asset["asset_id"] for asset in result_pa01["matched_assets"]
    ],
    "schema_valid": result_pa01["schema_valid"],
    "evidence_valid": result_pa01["evidence_valid"],
}
pprint(summary_pa01)

{'evidence_valid': True,
 'matched_asset_ids': ['A-REM-01', 'A-REM-02', 'C-REM-01'],
 'processing_status': 'DONE',
 'reason_codes': ['PRODUCT_AND_VERSION_MATCH'],
 'request_id': 'REQ-PY-0001',
 'schema_valid': True,
 'status': 'POTENTIALLY_AFFECTED'}


In [10]:
assert result_pa01["processing_status"] == "DONE"
assert result_pa01["status"] == "POTENTIALLY_AFFECTED"
assert [
    asset["asset_id"] for asset in result_pa01["matched_assets"]
] == ["A-REM-01", "A-REM-02", "C-REM-01"]
assert result_pa01["schema_valid"] is True
assert result_pa01["evidence_valid"] is True

print("PA-01 stimmt mit dem bestätigten n8n-Ergebnis überein.")

PA-01 stimmt mit dem bestätigten n8n-Ergebnis überein.


## 10. Tests der Versionsregeln
## *10. Tests of the version rules*

Die sechs im Workflow vorgesehenen Regeltypen werden hier direkt geprüft. `unknown` liefert bewusst kein geratenes Ja/Nein.<br>
*The six rule types of the workflow are tested directly. `unknown` deliberately returns no guessed yes/no.*

In [11]:
def affected_rule(
    rule_type: str,
    *,
    lower: Optional[str] = None,
    upper: Optional[str] = None,
    lower_inclusive: Optional[bool] = None,
    upper_inclusive: Optional[bool] = None,
) -> dict[str, Any]:
    return {
        "version_rule_type": rule_type,
        "lower_bound": lower,
        "upper_bound": upper,
        "lower_inclusive": lower_inclusive,
        "upper_inclusive": upper_inclusive,
        "affected_version_raw": None,
    }


assert version_matches(
    "7.4",
    affected_rule("exact", lower="7.4", lower_inclusive=True),
) is True
assert version_matches(
    "2.3.1",
    affected_rule(
        "range",
        lower="2.3.0",
        upper="2.3.2",
        lower_inclusive=True,
        upper_inclusive=True,
    ),
) is True
assert version_matches(
    "5.6.9",
    affected_rule("before", upper="5.7.0", upper_inclusive=False),
) is True
assert version_matches(
    "3.8",
    affected_rule("through", upper="3.8", upper_inclusive=True),
) is True
assert version_matches("99.0", affected_rule("all")) is True
assert version_matches("1.0", affected_rule("unknown")) is None
assert compare_versions("2.3", "2.3.0") == 0

print("Alle sechs Versionsregeln verhalten sich wie vorgesehen.")

Alle sechs Versionsregeln verhalten sich wie vorgesehen.


## 11. Tests der vier fachlichen Statuspfade
## *11. Tests of the four status paths*

Diese kleinen strukturierten Fälle isolieren die Regelentscheidung vom LLM. Dadurch lässt sich prüfen, ob nicht das Modell, sondern tatsächlich die festen Regeln den Status setzen.<br>
*These small structured cases isolate the rule decision from the LLM. They show that the fixed rules, not the model, set the status.*

In [12]:
PRODUCT_TEMPLATE = {
    "vendor": None,
    "product": None,
    "affected_version_raw": None,
    "version_rule_type": "unknown",
    "lower_bound": None,
    "lower_inclusive": None,
    "upper_bound": None,
    "upper_inclusive": None,
    "fixed_version": None,
}


def base_facts(
    *,
    is_advisory: bool,
    product: Optional[dict[str, Any]],
    evidence: str,
    missing_information: Optional[list[str]] = None,
) -> dict[str, Any]:
    return {
        "is_security_advisory": is_advisory,
        "advisory_reference": None,
        "cve_ids": [],
        "affected_products": [] if product is None else [product],
        "severity_stated_by_source": None,
        "recommended_action": None,
        "missing_information": missing_information or [],
        "evidence_quotes": [evidence],
    }


def run_structured_case(facts: dict[str, Any]) -> dict[str, Any]:
    local_repository = InMemoryRepository()
    local_repository.accept(
        source_channel="form",
        payload_type="structured",
        raw_text=json.dumps(facts, ensure_ascii=False),
    )
    return process_next_request(
        local_repository,
        assets=ASSET_INVENTORY,
    )


no_match_facts = base_facts(
    is_advisory=True,
    product={
        **PRODUCT_TEMPLATE,
        "vendor": "LogicGate",
        "product": "LogicGate Control Center",
        "affected_version_raw": "4.1.0 bis 4.1.6",
        "version_rule_type": "range",
        "lower_bound": "4.1.0",
        "lower_inclusive": True,
        "upper_bound": "4.1.6",
        "upper_inclusive": True,
    },
    evidence="LogicGate Control Center",
)

review_facts = base_facts(
    is_advisory=True,
    product={
        **PRODUCT_TEMPLATE,
        "vendor": "Musterwerk Systems GmbH",
        "product": "MusterLink Remote Gateway",
    },
    evidence="MusterLink Remote Gateway",
    missing_information=["Betroffene Versionen sind nicht bekannt."],
)

invalid_facts = base_facts(
    is_advisory=False,
    product=None,
    evidence="keine Produktschwachstelle",
)

results = {
    "POTENTIALLY_AFFECTED": result_pa01,
    "NO_MATCH_FOUND": run_structured_case(no_match_facts),
    "REVIEW_REQUIRED": run_structured_case(review_facts),
    "INVALID_ADVISORY": run_structured_case(invalid_facts),
}

for expected_status, result in results.items():
    assert result["status"] == expected_status
    print(
        f"{expected_status:22s} | "
        f"{', '.join(result['reason_codes'])}"
    )

POTENTIALLY_AFFECTED   | PRODUCT_AND_VERSION_MATCH
NO_MATCH_FOUND         | PRODUCT_NOT_IN_INVENTORY
REVIEW_REQUIRED        | VERSION_INFORMATION_UNCLEAR
INVALID_ADVISORY       | NOT_A_SECURITY_ADVISORY


## 12. Technischer Fehlerpfad
## *12. Technical error path*

Wenn die LLM-Extraktion technisch scheitert, setzt der Workflow **keinen fachlichen Status**. Der Datensatz bleibt mit `processing_status = ERROR` und `error_detail` gespeichert und kann später wiederholt werden.<br>
*If the LLM extraction fails technically, the workflow sets __no status__. The record stays stored with `processing_status = ERROR` and `error_detail` and can be processed again later.*

In [13]:
error_repository = InMemoryRepository()
error_repository.accept(
    source_channel="email",
    payload_type="unstructured",
    raw_text="Synthetischer Text ohne bereitgestelltes LLM-Fixture.",
)

error_result = process_next_request(
    error_repository,
    assets=ASSET_INVENTORY,
    llm_outputs={},
)

assert error_result["processing_status"] == "ERROR"
assert error_result.get("status") is None
pprint(
    {
        "request_id": error_result["request_id"],
        "processing_status": error_result["processing_status"],
        "error_detail": error_result["error_detail"],
    }
)

{'error_detail': 'LLM extraction failed: Für den Offline-Lauf fehlt ein '
                 'LLM-Fixture.',
 'processing_status': 'ERROR',
 'request_id': 'REQ-PY-0001'}


## 13. Was im Produktionsbetrieb anders wäre
## *13. What would be different in production*

Der Notebook-Speicher dient nur dem Verständnis. In PostgreSQL übernimmt n8n den ältesten offenen Datensatz atomar:<br>
*The in-memory store is only for understanding. In PostgreSQL, n8n claims the oldest open record atomically:*

```sql
WITH next_request AS (
    SELECT request_id
    FROM public.advisory_requests
    WHERE processing_status = 'NEW'
    ORDER BY received_at ASC, request_id ASC
    FOR UPDATE SKIP LOCKED
    LIMIT 1
)
UPDATE public.advisory_requests AS ar
SET
    processing_status = 'PROCESSING',
    attempt_count = ar.attempt_count + 1
FROM next_request
WHERE ar.request_id = next_request.request_id
RETURNING ar.*;
```

`FOR UPDATE SKIP LOCKED` verhindert, dass zwei parallele Worker denselben Datensatz verarbeiten. Weitere Unterschiede:<br>
*`FOR UPDATE SKIP LOCKED` prevents two parallel workers from processing the same record. Further differences:*

- Credentials bleiben in n8n beziehungsweise einem Secret Store.<br>
  *Credentials stay in n8n or a secret store.*
- Der Webhook braucht Authentifizierung.<br>
  *The webhook needs authentication.*
- Asset-Daten kommen aus PostgreSQL, CMDB oder RMM statt aus einer Python-Liste.<br>
  *Asset data comes from PostgreSQL, a CMDB or an RMM instead of a Python list.*
- Ein Produktivsystem benötigt Retry-Regeln, Monitoring, Berechtigungen und Regressionstests.<br>
  *A production system needs retry rules, monitoring, permissions and regression tests.*
- `POTENTIALLY_AFFECTED` bleibt ein Prüfhinweis, keine automatische Patch-Freigabe.<br>
  *`POTENTIALLY_AFFECTED` remains a hint for review, not an automatic patch approval.*
- `NO_MATCH_FOUND` bedeutet nur: kein Treffer im vorhandenen Inventar.<br>
  *`NO_MATCH_FOUND` only means: no match in the existing inventory.*

## 14. Ergebnis
## *14. Result*

Das Notebook bildet den abgeschlossenen Prozess von Eingang bis gespeicherter Relevanzentscheidung nach:<br>
*The notebook reproduces the closed process from input to stored decision:*

`NEW → PROCESSING → Extraktion → Validierung → Asset-Abgleich → Status → DONE`<br>
*`NEW → PROCESSING → extraction → validation → asset matching → status → DONE`*

Bei einem technischen Modellfehler gilt stattdessen:<br>
*On a technical model error instead:*

`NEW → PROCESSING → ERROR`

Die fachlich wichtigste Grenze bleibt sichtbar: Das LLM liefert strukturierte Fakten. Die Entscheidungen über Produktvergleich, Versionsbereich, Status und Weiterleitung werden deterministisch getroffen.<br>
*The most important boundary stays visible: the LLM delivers structured facts. Product comparison, version range, status and routing are decided deterministically.*